# Identifying Common Visual Themes in an Annual Report (Gemini API)

Obaid and Pukthuanthong ([2022](https://doi.org/10.1016/j.jfineco.2021.06.002), *Journal of Financial Economics*) train a convolutional neural network to classify the sentiment (positive/negative) of a large sample of Wall Street Journal news photos, and use the daily proportion of negative photos — *Photo Pessimism* — as a market-level investor sentiment index. They show it predicts short-horizon return reversals, is strongest during periods of elevated fear, and captures information distinct from (and substituting for) the sentiment embedded in the accompanying article text. Their approach is a bespoke, purpose-trained image classifier requiring a hand-labeled training set; the exercise below instead uses a general-purpose multimodal LLM (Gemini) to extract structured information — subject, setting, and eventually themes — directly from images with no training data of your own required, a much lower-cost (if less battle-tested) way to turn a collection of images into research data.

In [ ]:
import base64
import os
from pathlib import Path

import pymupdf
from dotenv import load_dotenv
from google import genai
from pydantic import BaseModel

load_dotenv()

### 1. Extract and filter embedded images

Not every embedded image in a PDF is meaningful content — this report also embeds signatures and small graphic elements alongside its photographs. Sorting all embedded images by pixel area shows a clean gap: everything under ~85,000 px² turns out to be a signature or small graphic, and everything over ~125,000 px² is a genuine photograph. We filter on that gap.

In [ ]:
PDF_PATH = Path("../data/BHP_Annual_report_2025.pdf")
IMAGE_DIR = Path("../data/BHP_images")
IMAGE_DIR.mkdir(parents=True, exist_ok=True)

MIN_AREA = 100_000  # px^2; separates real photos from signatures/icons in this report

doc = pymupdf.open(PDF_PATH)

images = []
for page_number in range(doc.page_count):
    for img in doc[page_number].get_images(full=True):
        xref = img[0]
        base = doc.extract_image(xref)
        width, height = base["width"], base["height"]
        if width * height < MIN_AREA:
            continue

        path = IMAGE_DIR / f"p{page_number + 1:03d}_{xref}.{base['ext']}"
        path.write_bytes(base["image"])
        images.append({"page": page_number + 1, "path": path, "width": width, "height": height})

print(f"Extracted {len(images)} photos from {doc.page_count} pages")

### 2. Caption each photo with Gemini

For each extracted photo, we ask Gemini for a short description of its subject and setting. Annual reports also feature plenty of posed headshots of directors and executives — these would otherwise dominate the theme summary with an unhelpful "portrait photo" result, so we also ask Gemini to flag headshots as it captions each photo, and exclude them before the theme-summary step. This loop makes one API call per photo — simple to follow, and fast enough for the ~40 photos in this report.

In [ ]:
client = genai.Client(api_key=os.getenv("GOOGLE_GAI_API"))

In [ ]:
CAPTION_PROMPT = (
    "Look at this photo from a mining company's annual report.\n"
    'Line 1: answer exactly "HEADSHOT" if this is a posed headshot/portrait photo of a person '
    '(e.g. a director or executive), or exactly "SCENE" otherwise.\n'
    "Line 2: a one- or two-sentence description of the subject and setting."
)

captions = []
n_headshots = 0
for i, image in enumerate(images):
    image_bytes = image["path"].read_bytes()
    mime_type = "image/png" if image["path"].suffix == ".png" else "image/jpeg"

    interaction = client.interactions.create(
        model="gemini-3.5-flash",
        input=[
            {"type": "text", "text": CAPTION_PROMPT},
            {"type": "image", "mime_type": mime_type, "data": base64.b64encode(image_bytes).decode("utf-8")},
        ],
    )

    label, _, caption = interaction.output_text.strip().partition("\n")
    if label.strip().upper() == "HEADSHOT":
        n_headshots += 1
        print(f"[{i + 1}/{len(images)}] p{image['page']}: (headshot, excluded)")
        continue

    caption = caption.strip()
    captions.append(caption)
    print(f"[{i + 1}/{len(images)}] p{image['page']}: {caption}")

print(f"\nExcluded {n_headshots} manager headshots; kept {len(captions)} scene photos for theme analysis")

### 3. Summarize the top five common themes

Rather than re-sending all ~40 photos, we feed Gemini the text captions we already collected and ask it to identify recurring themes — much cheaper, and this time we constrain the output to a schema (Structured Outputs) so we get back a clean, parseable list rather than free-form prose.

In [ ]:
class Theme(BaseModel):
    theme: str
    description: str

class ThemeSummary(BaseModel):
    top_themes: list[Theme]

captions_block = "\n".join(f"{i + 1}. {caption}" for i, caption in enumerate(captions))

interaction = client.interactions.create(
    model="gemini-3.5-flash",
    input=(
        f"Here are descriptions of {len(captions)} photos from a mining company's annual report:\n\n"
        f"{captions_block}\n\n"
        "Your task: Identify the five most common visual themes across these photos, with a short description of each."
    ),
    response_format={
        "type": "text",
        "mime_type": "application/json",
        "schema": ThemeSummary.model_json_schema(),
    },
)

summary = ThemeSummary.model_validate_json(interaction.output_text)
for theme in summary.top_themes:
    print(f"- {theme.theme}: {theme.description}")